# 常用类型与类型推断

学习目标：能用常见类型和推断表达数据，理解特殊类型的范围，并辨别断言与运行时转换、检查的区别。

前置知识：JavaScript 原始值与对象、函数、数组、条件判断；会用项目内 tsc 检查和生成代码。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/02-types-and-inference/。

1. [basics.ts](scripts/02-types-and-inference/basics.ts)：常用类型与推断；[assertions.ts](scripts/02-types-and-inference/assertions.ts)：断言与转换的对比。
2. [type-errors.ts](scripts/02-types-and-inference/type-errors.ts)：单独检查的类型错误。
3. [any-runtime-error.ts](scripts/02-types-and-inference/any-runtime-error.ts)、[assertion-runtime-error.ts](scripts/02-types-and-inference/assertion-runtime-error.ts)、[nonnull-runtime-error.ts](scripts/02-types-and-inference/nonnull-runtime-error.ts)：类型检查放行、运行时失败的反例。
4. [tsconfig.json](scripts/02-types-and-inference/tsconfig.json)、[tsconfig.errors.json](scripts/02-types-and-inference/tsconfig.errors.json)：正常项目与预期类型错误项目。

Step 1：检查类型。

```bash
npm run check:02
```

Step 2：生成 JavaScript。

```bash
npm run build:02
```

Step 3：运行正常示例。

```bash
npm run run:02
# 依次运行 basics.js 与 assertions.js，预期正常退出。
```

类型错误文件不进入正常项目。三个运行时反例会参与类型检查和生成，但需在后文单独运行。

## 1 常用类型总览

TypeScript 的类型用于静态检查。下面既有对应 JavaScript 原始值的类型，也有表达检查范围的特殊类型，不是新增了一组 JavaScript 运行时类型。

| 类型名称 | 中文名称／含义 | 用途或范围 |
| --- | --- | --- |
| string | 字符串类型 | 原始字符串 |
| number | 数值类型 | JavaScript 数值，包括整数与小数 |
| boolean | 布尔类型 | true 或 false |
| bigint | 大整数类型 | 例如 12n |
| symbol | 符号类型 | 符号值 |
| unique symbol | 唯一符号类型 | 区分某一个符号的身份 |
| null | 空值类型 | null |
| undefined | 未定义值类型 | undefined |
| any | 任意类型 | 放宽检查，允许许多未经验证的操作 |
| unknown | 未知类型 | 可接收任意值，使用前通常需要确认具体类型 |
| never | 不可能出现的值类型 | 无可取的值，也用于无正常返回的函数 |
| void | 无可用返回值的类型 | 表示不依赖函数的返回值 |
| object | 非原始值类型 | 对象、数组、函数等 |
| Object | Object 接口类型 | 不能用来限制“只接受对象” |
| {} | 不包含 null 和 undefined 的类型 | strict 下也接受 0、false、空字符串 |

日常标注使用小写 string、number、boolean；大写 String、Number、Boolean 是包装对象类型，不应用来代替原始类型。Object 与 object 的范围也不同，下文会对比。

## 2 类型标注与类型推断

类型标注写在名称之后，例如 title: string。能从初始值清楚得出类型时，通常省略标注，由编译器进行类型推断（type inference）。

let count = 1 推断为 number；const courseName 的绑定不再改变，因而可以推断为更具体的字符串字面量类型。后者只表示这一个字符串值。

```typescript
let count = 1; // 推断为 number
const courseName = "TypeScript"; // 推断为字符串字面量类型 "TypeScript"
let title: string = "课程";
let completed: boolean = false;
let largeCount: bigint = 12n;
console.log(count, courseName, title, completed, largeCount); // 1 TypeScript 课程 false 12n
```

TypeScript 中 let 允许重新赋值，但新值仍需符合已确定的类型。下面是 [type-errors.ts](scripts/02-types-and-inference/type-errors.ts) 中的反例：

```typescript
let inferredCount = 1;
inferredCount = "1"; // TS2322：推断为 number 后不能再赋 string
```

## 3 类型位置与值位置

类型位置用于描述数据的约束；值位置需要程序实际执行时存在的值。下面冒号后的 number 是类型，等号后的 3 是值。

```typescript
const amount: number = 3;
console.log(amount, typeof amount); // 3 number：这里的 typeof 是 JavaScript 运行时运算符
```

不能把只存在于类型位置的 number 当作运行时变量：

```typescript
console.log(number); // TS2693：number 是类型，不能作为这里的运行时值
```

## 4 symbol 与 unique symbol

symbol 可以表示任意符号值；unique symbol 进一步记录某一个符号的身份。用变量声明 unique symbol 时需要 const。描述文本相同，也不代表两次 Symbol() 调用得到同一个符号。

下面的 typeof token 位于类型位置，取得 token 的静态类型；它不会执行 JavaScript 的 typeof 运算。

```typescript
const token: unique symbol = Symbol("token");
const sameToken: typeof token = token; // 类型位置的 typeof 取得 token 的静态类型
let generalToken: symbol = token;
console.log(sameToken === token, typeof generalToken); // true symbol
```

在反例中，第二次调用产生另一个符号，不能赋给要求首个符号身份的变量：

```typescript
const uniqueToken: unique symbol = Symbol("token");
const differentToken: typeof uniqueToken = Symbol("token"); // TS2322：另一个符号不具有同一身份
```

## 5 null 与 undefined

strict 包含严格空值检查。在本章配置下，null 和 undefined 不会自动成为 string、number 等类型的一部分。需要允许空值时，用联合类型明确写出。

```typescript
let absent: null = null;
let notSet: undefined = undefined;
let nickname: string | null = null; // | 表示联合：这里允许字符串或 null
nickname = "小林";
console.log(absent, notSet, nickname); // null undefined 小林
```

```typescript
const strictNumber: number = null; // TS2322：strict 下 number 不包含 null
```

## 6 any 与 unknown

any 会放宽检查，允许直接访问未确认的属性和调用方法；错误因此可能推迟到运行时。unknown 也可以接收任意值，但不会仅凭这个标注就允许调用字符串方法。

先用 typeof 确认值是字符串，编译器才在对应分支内把类型缩小为 string，这称为类型收窄（narrowing）。

```typescript
const externalValue: unknown = "course";
if (typeof externalValue === "string") {
  // 分支内已确认是字符串，可以使用字符串方法。
  console.log(externalValue.toUpperCase()); // COURSE
}
```

未收窄的反例：

```typescript
const unchecked: unknown = "text";
unchecked.toUpperCase(); // TS18046：尚未确认 unknown 的具体类型
```

any 则会让下例通过检查，但不能让数值获得字符串方法。该文件不在正常运行入口中：

```typescript
const flexible: any = 12;
flexible.toUpperCase(); // 类型检查放行；运行时 TypeError：数值没有这个字符串方法
```

```bash
node .build/02-types-and-inference/any-runtime-error.js
# 预期非零退出并出现 TypeError。
```

## 7 void 与 never

- void 用于表示调用方不依赖函数的返回值。它不是 undefined 的别名，也不承诺所有被标为 void 的调用在运行时一定返回 undefined。
- never 表示不可能出现的值；函数若总是抛出异常或永不结束，就没有正常返回的路径。

```typescript
function announce(message: string): void {
  console.log(message);
}
announce("已保存"); // 已保存

function fail(message: string): never {
  throw new Error(message); // 函数没有正常返回的路径；本正常示例不调用它
}
```

不能给 never 类型提供一个普通值：

```typescript
const impossible: never = 1; // TS2322：数值不能赋给 never
```

## 8 object、Object 与 {}

- object 排除原始值，适合只需要表达“这是某种对象”的位置。
- Object 是大写的接口类型，除 null、undefined 外的原始值也可赋给它，不能把它当作 object 的另一种拼写。
- {} 是没有列出成员的对象类型写法。在 strict 下，它允许除 null、undefined 以外的值，并不表示“只能传入空对象”。

```typescript
let objectValue: object = { score: 80 };
objectValue = [1, 2]; // 数组也是对象
let broadObject: Object = 1;
broadObject = "文本"; // Object 也允许这里的数值和字符串
let nonNullValue: {} = 0;
nonNullValue = false; // {} 不表示“只能是空对象”
console.log(Array.isArray(objectValue), broadObject, nonNullValue); // true 文本 false
```

下面这些赋值不成立：

```typescript
const onlyObject: object = 1; // TS2322：object 不接受原始值
const capitalObject: Object = null; // TS2322：这里不允许 null
const notNullish: {} = undefined; // TS2322：这里不允许 undefined
```

已知对象有哪些属性时，应写具体对象类型；不要因为名字看起来通用，就用 Object 或 {} 代替数据结构。对象类型的详细写法在后续章节展开。

## 9 上下文类型

类型推断也会利用表达式所在的位置，这称为上下文类型（contextual typing）。下面数组元素是字符串，forEach 的回调形参 name 因此被推断为 string，无需再手写标注。

```typescript
const names = ["Ada", "Lin"]; // 推断为 string[]，即字符串数组
names.forEach((name) => {
  // forEach 为每个元素调用回调；name 从调用位置推断为 string。
  console.log(name.toUpperCase()); // 依次输出 ADA、LIN
});
```

如果在同样的回调中调用 name.toFixed(2)，会得到 TS2551：string 没有这个数值格式化方法。类型来自调用位置，不是形参名称 name。

## 10 类型断言不转换值

类型断言（type assertion）用 as 告诉编译器按另一种允许的类型关系看待表达式。断言在输出中被擦除，不会转换值，也不会验证它是否真的符合目标类型。

```typescript
const input: unknown = 42;
const assumedText = input as string; // 只改变编译器对这个表达式的看法
console.log(typeof assumedText); // number：运行时的值仍是数值
const convertedText = String(input); // 真正执行字符串转换
console.log(typeof convertedText); // string
```

如果继续把这个数值当字符串调用，运行仍会失败。反例见 [assertion-runtime-error.ts](scripts/02-types-and-inference/assertion-runtime-error.ts)：

```typescript
const input: unknown = 12;
const text = input as string;
text.toUpperCase(); // 类型检查通过；运行时 TypeError：断言没有转换数值
```

```bash
node .build/02-types-and-inference/assertion-runtime-error.js
# 预期非零退出并出现 TypeError。
```

断言也不是任意类型之间都能直接使用；应优先用推断、收窄或真实的数据转换表达意图。这里从 unknown 断言为 string，是为了观察断言的边界。

## 11 非空断言不检查空值

非空断言（non-null assertion）写成表达式后的 !，从静态类型中排除 null 和 undefined。它不是逻辑取反，也不会插入运行时的空值检查。

```typescript
function titleLength(title: string | null): number {
  return title!.length; // ! 从静态类型中排除 null 和 undefined，不检查实际值
}
console.log(titleLength("JS")); // 2：这次实参确实是字符串
```

传入 null 的反例见 [nonnull-runtime-error.ts](scripts/02-types-and-inference/nonnull-runtime-error.ts)：

```typescript
function titleLength(title: string | null): number {
  return title!.length;
}
titleLength(null); // 类型检查通过；运行时 TypeError：! 没有检查 null
```

```bash
node .build/02-types-and-inference/nonnull-runtime-error.js
# 预期非零退出并出现 TypeError。
```

函数允许传入 null，却直接读取其属性，说明这个断言缺少依据。实际代码应先判断是否为空，再访问属性；或根据需求修改参数类型。

## 12 集中检查类型反例

[type-errors.ts](scripts/02-types-and-inference/type-errors.ts) 集中了本章静态反例，另含包装对象与原始字符串的对比：

```typescript
const primitiveText: string = new String("x"); // TS2322：包装对象不是原始字符串
```

```bash
npm run errors:02
# 预期非零退出；诊断包含 TS2322、TS2693、TS18046、TS2551。
# 逐个对照源码旁的注释，不执行这个类型错误文件。
```

该命令使用 tsconfig.errors.json，只检查预期类型错误，不影响 check:02。修改反例后，应核对错误原因是否消失，而不只是看错误数量是否减少。

## 本章小结

- 简单初始值与调用位置通常足以支持推断；类型位置和运行时值位置要分清。
- unknown 促使使用前确认类型；any、断言和非空断言不能替代运行时检查。
- object、Object、{} 的范围不同；strict 下是否允许 null 和 undefined 需要明确表达。

## 练习

1. 声明一个值可以为空的课程名，先判断是否为字符串，再调用 toUpperCase()；分别用字符串和 null 运行，确认都没有异常。
2. 改写 titleLength()：输入 null 时返回 0，其他情况下返回长度，不使用 ! 或 as。检查类型，并核对 null、空字符串、"JS" 三种输入。
3. 为数值 42 分别使用类型断言和 String() 转换，输出 typeof；说明为什么两者的运行结果不同。
4. 从 type-errors.ts 选择两个错误，保留原本需求修复后重新检查，确认对应行的诊断消失。

## 参考与引用来源

- TypeScript 官方文档：[Everyday Types](https://www.typescriptlang.org/docs/handbook/2/everyday-types.html) 中原始类型、推断、字面量、类型断言与非空断言；[Contextual Typing](https://www.typescriptlang.org/docs/handbook/type-inference.html#contextual-typing)；[Symbols / unique symbol](https://www.typescriptlang.org/docs/handbook/symbols.html#unique-symbol)；[More on Functions](https://www.typescriptlang.org/docs/handbook/2/functions.html#other-types-to-know-about) 中 void、object、unknown、never；[类型声明建议](https://www.typescriptlang.org/docs/handbook/declaration-files/do-s-and-don-ts.html#general-types) 中原始类型与 Object；[4.8 发布说明](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-8.html#improved-intersection-reduction-union-compatibility-and-narrowing) 中 {} 与空值的关系；[typeof 类型运算符](https://www.typescriptlang.org/docs/handbook/2/typeof-types.html)、[strictNullChecks](https://www.typescriptlang.org/tsconfig/strictNullChecks.html)：静态类型范围、推断和空值边界。
- MDN：[String()](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/String/String)、[typeof](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Operators/typeof)、[Array.prototype.forEach()](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/Array/forEach)：实际字符串转换、运行时类型标签与逐元素回调。